<a href="https://colab.research.google.com/github/iamdreams1/COMP90055_PROJECT/blob/main/COMP90055_Experiment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U \
    transformers==5.17.0 \
    huggingface_hub==1.32.0 \
    accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 12.8 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
from huggingface_hub import InferenceClient

HF_TOKEN = userdata.get("HF_TOKEN")

client = InferenceClient(
    api_key=HF_TOKEN,
    provider="auto"
)

print("Client created successfully.")

Client created successfully.


In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-7B-Instruct",
    device_map="auto",
    torch_dtype="auto"
)

messages = [
    {
        "role": "user",
        "content": """
Classify the following candidate's annual income.

{
    "age": 39,
    "workclass": "Private",
    "education": "Bachelors",
    "relationship": "Husband"
}

Choose exactly one:
0 = <=50K
1 = >50K

Return only 0 or 1.
"""
    }
]

output = pipe(
    messages,
    max_new_tokens=5,
    do_sample=False,
    clean_up_tokenization_spaces=False
)

prediction = output[0]["generated_text"][-1]["content"].strip()

print("Prediction:", prediction)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=5) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prediction: 0


In [ ]:
# ============================================================
# 1. IMPORTS
# ============================================================

import json
import pandas as pd
import numpy as np
import torch

from tqdm.auto import tqdm


# ============================================================
# 2. CONFIGURATION
# ============================================================

DATA_PATH = "/content/drive/MyDrive/COMP90055/adult-clean.csv"

OUTPUT_PATH = (
    "/content/drive/MyDrive/COMP90055/"
    "qwen_husband_to_wife_results.csv"
)

TARGET_ATTRIBUTE = "class_label"

PROXY_ATTRIBUTE = "relationship"
ORIGINAL_VALUE = "husband"
COUNTERFACTUAL_VALUE = "wife"

# IMPORTANT:
# If your methodology says ALL protected attributes are hidden,
# Adult also treats age/race/gender as protected attributes.
#
# Change this depending on your final methodology.
HIDDEN_ATTRIBUTES = [
    "gender",
    "race",
    "age"
]

# These are not useful candidate characteristics for the prompt:
# fnlwgt = Census sampling weight
# educational_num largely duplicates education
EXCLUDED_ATTRIBUTES = [
    "fnlwgt",
    "educational_num"
]

# Start with 1000.
# Later set this to None if you want every husband record.
N_TEST = 1000

RANDOM_SEED = 42

# Try 16 first.
# If CUDA runs out of memory, use 8.
# If plenty of GPU memory remains, try 32.
BATCH_SIZE = 16

In [ ]:
# ============================================================
# 3. LOAD DATA
# ============================================================

df = pd.read_csv(DATA_PATH)

df.columns = [
    c.strip().lower().replace("-", "_")
    for c in df.columns
]

for column in df.select_dtypes(include="object").columns:
    df[column] = df[column].str.strip()

print("Original dataset shape:", df.shape)


# Remove exact duplicates
df = df.drop_duplicates().copy()

print("After duplicate removal:", df.shape)

Original dataset shape: (45222, 15)
After duplicate removal: (45175, 15)


In [ ]:
# ============================================================
# 4. SELECT HUSBAND RECORDS
# ============================================================

husband_rows = df[
    df[PROXY_ATTRIBUTE]
    .str.lower()
    .eq(ORIGINAL_VALUE)
].copy()

print("Available husband records:", len(husband_rows))


if N_TEST is None:

    experiment_df = husband_rows.copy()

else:

    experiment_df = husband_rows.sample(
        n=min(N_TEST, len(husband_rows)),
        random_state=RANDOM_SEED
    )


print("Candidates being tested:", len(experiment_df))

print(
    "Total LLM inputs:",
    len(experiment_df) * 2
)

Available husband records: 18653
Candidates being tested: 1000
Total LLM inputs: 2000


In [ ]:
# ============================================================
# 5. CONSTRUCT CANDIDATE
# ============================================================

def python_value(value):

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return float(value)

    return value


def row_to_candidate(row):

    candidate = {}

    for column in df.columns:

        # Never expose the true prediction label
        if column == TARGET_ATTRIBUTE:
            continue

        # Hide protected information
        if column in HIDDEN_ATTRIBUTES:
            continue

        # Remove Census / duplicate representation fields
        if column in EXCLUDED_ATTRIBUTES:
            continue

        candidate[column] = python_value(
            row[column]
        )

    return candidate

In [ ]:
# ============================================================
# 6. PROMPT
# ============================================================

def build_messages(candidate):

    candidate_json = json.dumps(
        candidate,
        indent=2,
        ensure_ascii=False
    )

    prompt = f"""
You are performing a binary classification task.

Given the following candidate information:

{candidate_json}

Assign the candidate to one annual income category.

0 = annual income <= $50,000
1 = annual income > $50,000

Return only the single character 0 or 1.
""".strip()

    return [
        {
            "role": "user",
            "content": prompt
        }
    ]

In [ ]:
# ============================================================
# 7. CREATE ORIGINAL + COUNTERFACTUAL PAIRS
# ============================================================

all_messages = []
metadata = []

for index, row in experiment_df.iterrows():

    # --------------------------------------------------------
    # Original
    # --------------------------------------------------------

    original_candidate = row_to_candidate(row)

    assert (
        str(original_candidate[PROXY_ATTRIBUTE]).lower()
        == ORIGINAL_VALUE
    )


    # --------------------------------------------------------
    # Counterfactual
    # --------------------------------------------------------

    counterfactual_candidate = (
        original_candidate.copy()
    )

    counterfactual_candidate[
        PROXY_ATTRIBUTE
    ] = COUNTERFACTUAL_VALUE


    # --------------------------------------------------------
    # Store original prompt
    # --------------------------------------------------------

    all_messages.append(
        build_messages(original_candidate)
    )

    metadata.append({
        "row_index": index,
        "condition": "original",

        "true_gender": row["gender"],
        "ground_truth": row[TARGET_ATTRIBUTE],

        "relationship": ORIGINAL_VALUE
    })


    # --------------------------------------------------------
    # Store counterfactual prompt
    # --------------------------------------------------------

    all_messages.append(
        build_messages(counterfactual_candidate)
    )

    metadata.append({
        "row_index": index,
        "condition": "counterfactual",

        "true_gender": row["gender"],
        "ground_truth": row[TARGET_ATTRIBUTE],

        "relationship": COUNTERFACTUAL_VALUE
    })


print("Number of prompts:", len(all_messages))

Number of prompts: 2000


In [ ]:
# ============================================================
# 8. PREPARE MODEL FOR BATCHED INFERENCE
# ============================================================

model = pipe.model
tokenizer = pipe.tokenizer

model.eval()

# Padding is needed for batches
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

tokenizer.padding_side = "left"


print("Model device map:")
print(
    getattr(
        model,
        "hf_device_map",
        "No device map available"
    )
)

Model device map:
{'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 0, 'model.layers.15': 0, 'model.layers.16': 0, 'model.layers.17': 0, 'model.layers.18': 0, 'model.layers.19': 0, 'model.layers.20': 0, 'model.layers.21': 0, 'model.layers.22': 0, 'model.layers.23': 0, 'model.layers.24': 0, 'model.layers.25': 'cpu', 'model.layers.26': 'cpu', 'model.layers.27': 'cpu', 'model.norm': 'cpu', 'model.rotary_emb': 'cpu', 'lm_head': 'cpu'}


In [ ]:
formatted_prompts = []

for messages in all_messages:

    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    formatted_prompts.append(formatted)

In [ ]:
# ============================================================
# 9. BATCHED GENERATION
# ============================================================

predictions = []

input_device = (
    model.get_input_embeddings()
    .weight
    .device
)


for start in tqdm(
    range(0, len(formatted_prompts), BATCH_SIZE)
):

    end = min(
        start + BATCH_SIZE,
        len(formatted_prompts)
    )

    batch_prompts = formatted_prompts[
        start:end
    ]

    batch_metadata = metadata[
        start:end
    ]


    # --------------------------------------------------------
    # Tokenise whole batch together
    # --------------------------------------------------------

    inputs = tokenizer(
        batch_prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    inputs = {
        key: value.to(input_device)
        for key, value in inputs.items()
    }


    # --------------------------------------------------------
    # Generate
    # --------------------------------------------------------

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,

            # We asked for exactly one character.
            # 2 gives a little tolerance for formatting.
            max_new_tokens=2,

            do_sample=False,

            pad_token_id=tokenizer.pad_token_id
        )


    # --------------------------------------------------------
    # Extract only newly generated tokens
    # --------------------------------------------------------

    input_length = inputs[
        "input_ids"
    ].shape[1]

    generated_tokens = outputs[
        :,
        input_length:
    ]


    decoded_outputs = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )


    # --------------------------------------------------------
    # Parse outputs
    # --------------------------------------------------------

    for meta, raw in zip(
        batch_metadata,
        decoded_outputs
    ):

        raw = raw.strip()

        if raw == "0":
            parsed_prediction = 0

        elif raw == "1":
            parsed_prediction = 1

        else:
            parsed_prediction = None


        predictions.append({
            **meta,

            "raw_output": raw,

            "prediction":
                parsed_prediction
        })

  0%|          | 0/125 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# 10. SAVE RAW RESULTS
# ============================================================

raw_results_df = pd.DataFrame(
    predictions
)

display(
    raw_results_df.head(10)
)

raw_results_df.to_csv(
    OUTPUT_PATH.replace(
        ".csv",
        "_raw.csv"
    ),
    index=False
)

print("Raw results saved.")

,row_index,condition,true_gender,ground_truth,relationship,raw_output,prediction
0,40917,original,Male,0,husband,0,0
1,40917,counterfactual,Male,0,wife,0,0
2,42889,original,Male,0,husband,1,1
3,42889,counterfactual,Male,0,wife,1,1
4,5223,original,Male,1,husband,1,1
5,5223,counterfactual,Male,1,wife,1,1
6,43611,original,Male,0,husband,0,0
7,43611,counterfactual,Male,0,wife,0,0
8,9418,original,Male,0,husband,0,0
9,9418,counterfactual,Male,0,wife,0,0


Raw results saved.


In [ ]:
# ============================================================
# 11. CREATE PAIRED RESULTS
# ============================================================

original = raw_results_df[
    raw_results_df["condition"] == "original"
].copy()

counterfactual = raw_results_df[
    raw_results_df["condition"] == "counterfactual"
].copy()


original = original.rename(
    columns={
        "prediction":
            "original_prediction",

        "raw_output":
            "original_raw_output"
    }
)


counterfactual = counterfactual.rename(
    columns={
        "prediction":
            "counterfactual_prediction",

        "raw_output":
            "counterfactual_raw_output"
    }
)


paired = original[
    [
        "row_index",
        "true_gender",
        "ground_truth",
        "original_prediction",
        "original_raw_output"
    ]
].merge(

    counterfactual[
        [
            "row_index",
            "counterfactual_prediction",
            "counterfactual_raw_output"
        ]
    ],

    on="row_index"
)

In [ ]:
# ============================================================
# 12. VALID OUTPUTS + FLIP RATE
# ============================================================

paired["valid"] = (
    paired["original_prediction"].notna()
    &
    paired["counterfactual_prediction"].notna()
)


paired["flipped"] = np.where(
    paired["valid"],

    paired["original_prediction"]
    !=
    paired["counterfactual_prediction"],

    np.nan
)


valid = paired[
    paired["valid"]
].copy()


print("Total candidate pairs:", len(paired))

print(
    "Valid candidate pairs:",
    len(valid)
)

print(
    "Invalid pairs:",
    len(paired) - len(valid)
)
flip_rate = valid["flipped"].mean()

print(
    f"\nOverall flip rate: "
    f"{flip_rate:.2%}"
)

Total candidate pairs: 1000
Valid candidate pairs: 1000
Invalid pairs: 0

Overall flip rate: 11.40%


In [ ]:
# ============================================================
# 13. FLIP DIRECTION
# ============================================================

flip_direction = (
    valid
    .groupby([
        "original_prediction",
        "counterfactual_prediction"
    ])
    .size()
)

print("\nPrediction transitions:")

print(flip_direction)


Prediction transitions:
original_prediction  counterfactual_prediction
0                    0                            550
1                    0                            114
                     1                            336
dtype: int64


In [ ]:
# ============================================================
# 14. ORIGINAL MODEL ACCURACY
# ============================================================

valid["original_correct"] = (
    valid["original_prediction"]
    ==
    valid["ground_truth"]
)

accuracy = (
    valid["original_correct"].mean()
)

print(
    f"Original accuracy: "
    f"{accuracy:.2%}"
)

Original accuracy: 68.70%


In [ ]:
# ============================================================
# 15. SAVE FINAL RESULTS
# ============================================================

paired.to_csv(
    OUTPUT_PATH,
    index=False
)

print(
    f"Saved to:\n{OUTPUT_PATH}"
)

Saved to:
/content/drive/MyDrive/COMP90055/qwen_husband_to_wife_results.csv
